# Quick inspect

## Set-up

### Imports

In [1]:
import pylbsr.notebooks
import pylbsr.misc

import json
import torch
import yaml
import pandas as pd
from functools import partial
from pathlib import Path
from tqdm import tqdm

### Parameters

In [2]:
_notebook_name = "quick_inspect.py.ipynb"
_notebook_path = f"notebooks/diagnostics/{_notebook_name}"


### Initialization

In [3]:
pylbsr.notebooks.enable_cell_timing_metadata(show=True)

logger = pylbsr.misc.init_logger(_notebook_name)

PROJECT_DIR = pylbsr.notebooks.find_project_root_from_notebook_path(_notebook_path)
logger.info(f"Project directory: {PROJECT_DIR}")

[11:19:15] INFO - Project directory: /home/l10n/projects/parnet-project/parnet--demo/parnet--demo--train-models


## Misc

### Dataset Metadata

In [13]:
# Load the metadata of the dataset

_metadata_fp = str(PROJECT_DIR / "resources/parnet-encore-eclip/600nt_windows.no-one-hot.stripped/encode.filtered.hfds/encode.filtered.metadata.yaml")
with open(_metadata_fp, "r") as f:
    _metadata = yaml.safe_load(f)

display(_metadata)

{'is_pre_padded': False,
 'n_tracks': 223,
 'seq_len': 600,
 'sequence_format': 'string',
 'signal_format': 'sparse_list',
 'source': '/mnt/storage-nas-fast-2/research/projects/hzm/parnet-analyses/parnet-demo/parnet--demo--train-models/resources/parnet-encore-eclip/600nt_windows.no-one-hot.stripped/encode.filtered.pt',
 'splits': {'test': 70626, 'train': 512946, 'valid': 116542},
 'task_names': ['control', 'eCLIP'],
 'total_key': 'eCLIP'}

⏱ 0.02 s (00:00:00)


### Single element

In [8]:
# HFDS — streams one shard, no full load
# --------------------------------------

from datasets import load_from_disk

_hfds = load_from_disk(str(PROJECT_DIR / "resources/parnet-encore-eclip/600nt_windows.no-one-hot.stripped/encode.filtered.hfds"))
elem = _hfds["train"][0]

# .pt alternative (mmap, also low RAM but slightly slower first access)
# ---------------------------------------------------------------------
# import torch
# _data = torch.load(PROJECT_DIR / "resources/parnet-encore-eclip/600nt_windows.no-one-hot.stripped/encode.filtered.pt", mmap=True, weights_only=False)
# elem = _data["train"][0]

# -------

print("meta    :", elem["meta"])
print("seq len :", len(elem["inputs"]["sequence"]))
print("seq[:20]:", elem["inputs"]["sequence"][:20])
print("outputs :", {k: dict(size=v["size"]) for k, v in elem["outputs"].items()})


Loading dataset from disk:   0%|          | 0/76 [00:00<?, ?it/s]

Loading dataset from disk:   0%|          | 0/18 [00:00<?, ?it/s]

meta    : {'name': 'chr14:100374289-100374889:-', 'pad_side': -1}
seq len : 600
seq[:20]: ATCTTTCTTTTAGTGTTTAA
outputs : {'control': {'size': [223, 600]}, 'eCLIP': {'size': [223, 600]}}
⏱ 33.75 s (00:00:33)


### Verify padding status of a few elements

In [16]:
from parnet_demo_utils import parse_tile_name

WINDOW_LEN = 600
MAX_CHECK = 100

_split = _hfds["train"]

# .pt alternative:
# import torch
# _split = torch.load(PROJECT_DIR / "resources/.../encode.filtered.pt", mmap=True, weights_only=False)["train"]

# ── Check ────────────────────────────────────────────────────────────────────
n_checked = n_prepadded = n_stripped = n_unexpected = 0
examples = []

for i in (pbar := tqdm(range(len(_split)), total=min(len(_split), MAX_CHECK), desc="checking")):
    pbar.set_postfix(checked=n_checked, prepadded=n_prepadded, stripped=n_stripped)
    if n_checked >= MAX_CHECK:
        break

    elem = _split[i]
    _, start, end, _ = parse_tile_name(elem["meta"]["name"])
    genomic_len = end - start
    seq_len = len(elem["inputs"]["sequence"])

    if genomic_len >= WINDOW_LEN:
        continue  # full-window tile — uninformative, skip

    n_checked += 1
    if seq_len == WINDOW_LEN:
        n_prepadded += 1
    elif seq_len == genomic_len:
        n_stripped += 1
    else:
        n_unexpected += 1

    if len(examples) < 5:
        examples.append((elem, genomic_len, seq_len))

    if n_checked >= MAX_CHECK:
        break

print(f"Short tiles checked (genomic < {WINDOW_LEN}): {n_checked}")
print(f"  pre-padded  seq == {WINDOW_LEN}        : {n_prepadded}")
print(f"  stripped    seq == genomic_len : {n_stripped}")
print(f"  unexpected                     : {n_unexpected}")
print()
for name, g, s in examples:
    print(f"  {name['meta']['name']}  genomic={g}  seq={s}")



checking: 7740it [00:18, 413.18it/s, checked=99, prepadded=0, stripped=99]                     


Short tiles checked (genomic < 600): 100
  pre-padded  seq == 600        : 0
  stripped    seq == genomic_len : 100
  unexpected                     : 0

  chr16:69385827-69386007:-  genomic=180  seq=180
  chr17:82033927-82034204:+  genomic=277  seq=277
  chr12:104125235-104125356:+  genomic=121  seq=121
  chr17:4988138-4988580:+  genomic=442  seq=442
  chr17:41865167-41865423:-  genomic=256  seq=256
⏱ 18.74 s (00:00:18)


In [18]:
# Show that the shape of the outputs of a stripped tile matches the sequence length.
elem = examples[0][0]

print("meta    :", elem["meta"])
print("seq len :", len(elem["inputs"]["sequence"]))
print("seq[:20]:", elem["inputs"]["sequence"][:20])
print("outputs :", {k: dict(size=v["size"]) for k, v in elem["outputs"].items()})

meta    : {'name': 'chr16:69385827-69386007:-', 'pad_side': 0}
seq len : 180
seq[:20]: AGAAGCCCGCGGGCGGCTCG
outputs : {'control': {'size': [223, 180]}, 'eCLIP': {'size': [223, 180]}}
⏱ 0.00 s (00:00:00)
